# Malaysia monthly CPI trends

This notebook reads the official `cpi_headline` dataset from the DOSM OpenDOSM API. The dataset page describes `index` as an index with base 2010 = 100 and `date` as monthly. It is not an absolute household cost or a personal inflation measure.

Sources: [dataset metadata](https://open.dosm.gov.my/data-catalogue/cpi_headline), [OpenDOSM API](https://api.data.gov.my/opendosm?id=cpi_headline), and [API query syntax](https://developer.data.gov.my/request-query). Data is published under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The API requires no token and allows four requests per minute for the OpenDOSM API family.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

API_URL = "https://api.data.gov.my/opendosm"
params = {
    "id": "cpi_headline",
    "filter": "overall@division",
    "sort": "date",
    "limit": 1000,
    "meta": "true",
}
response = requests.get(API_URL, params=params, timeout=30)
response.raise_for_status()
payload = response.json()
metadata = payload["meta"]
cpi = pd.DataFrame(payload["data"])
cpi["date"] = pd.to_datetime(cpi["date"])
cpi = cpi.sort_values("date").reset_index(drop=True)

print("Dataset:", metadata["catalogue_id"])
print("Source:", metadata.get("data_source"))
print("Data as of:", metadata.get("data_as_of"))
print("Rows:", len(cpi), "Coverage:", cpi["date"].min().date(), "to", cpi["date"].max().date())
cpi.tail()

## Year-on-year change

For a monthly series, `pct_change(12)` compares each index with the same month one year earlier. The resulting percentage is derived from the source index and should not be interpreted as the price change experienced by every household.

In [ ]:
cpi["change_yoy_pct"] = cpi["index"].pct_change(periods=12) * 100
display(cpi[["date", "index", "change_yoy_pct"]].tail(18))

ax = cpi.plot(x="date", y="index", figsize=(11, 4), legend=False, color="#167f79")
ax.set_title("Malaysia overall CPI (2010 = 100)")
ax.set_xlabel("Month")
ax.set_ylabel("Index")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()